# Семинар 2. Подготовка данных для линейной регрессии

## Как использовать признаки с пропусками и названия районов?

В прошлом семинаре мы обучили модель по трём площадям. Теперь попробуем добавить сведения, которые пока не использовали: год постройки, длину границы участка вдоль улицы и район.

Появятся две задачи: в одном числовом столбце есть пропуски, а район записан строкой. Научимся готовить такие признаки и объединим подготовку с моделью в одну последовательность действий.

**План:** `fit` и `transform` → пропуски → масштабирование → категории → `Pipeline` и `ColumnTransformer` → сравнение прогнозов.

Выполняйте ячейки по порядку. Это самостоятельный ноутбук: запускать предыдущий перед ним не нужно. В конце — задания без готовых решений.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

sns.set(style="darkgrid")

## 1. Возвращаемся к домам

Используем ту же таблицу [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data). Целевая переменная — стоимость продажи `SalePrice`, в долларах США.

| Признак | Смысл | Единицы после подготовки |
|---|---|---|
| `GrLivArea` | Надземная жилая площадь | м² |
| `GarageArea` | Площадь гаража | м² |
| `TotalBsmtSF` | Общая площадь подвала | м² |
| `YearBuilt` | Год постройки | календарный год |
| `LotFrontage` | Длина границы участка вдоль улицы | м |
| `Neighborhood` | Район города | название-код района |
| `LotArea` | Площадь участка, понадобится в задании | м² |

`LotFrontage` — длина, а не площадь. Её переводим из футов в метры, умножая на 0,3048. Для площадей сохраняем множитель 0,09290304. Названия столбцов оставляем прежними.

In [ ]:
DATA_URL = (
    "https://www.dropbox.com/scl/fi/65u0zpc8dweb9rx1tlzol/train_sem2.csv"
    "?rlkey=5vi4mt47fs6magpldgj5wp2u9&dl=1"
)
raw_data = pd.read_csv(DATA_URL)
print(f"Загружено {len(raw_data)} домов.")

In [ ]:
BASE_FEATURES = ["GrLivArea", "GarageArea", "TotalBsmtSF"]
numeric_features = BASE_FEATURES + ["YearBuilt", "LotFrontage"]
categorical_features = ["Neighborhood"]
features = numeric_features + categorical_features

data = raw_data.copy()
data[BASE_FEATURES + ["LotArea"]] *= 0.09290304
data["LotFrontage"] *= 0.3048
assert (data["SalePrice"] > 0).all()

Сохраняем **те же дома в train, validation и test**, что и в первом семинаре: 60%, 20% и 20%. `random_state` и порядок строк не меняем.

* На **train** обучаем и подготовку признаков, и саму модель.
* На **validation** сравниваем варианты по метрикам.
* **Test** остаётся отложенным до окончательной проверки в конце модуля.

В этом занятии основная метрика выбора — MAE. RMSE и MAPE помогают рассмотреть ошибки с других сторон.

In [ ]:
train_val, test = train_test_split(data, test_size=0.2, random_state=42)
train, validation = train_test_split(train_val, test_size=0.25, random_state=42)

train = train.copy()
validation = validation.copy()
test = test.copy()
y_train = train["SalePrice"]
y_val = validation["SalePrice"]

print(f"Train: {len(train)}; validation: {len(validation)}; test: {len(test)}.")
print("Test отложен: сегодня его не исследуем и не оцениваем.")

In [ ]:
train[features + ["SalePrice"]].head()

### Повторим метрики

**MAE (Mean Absolute Error)** — средняя абсолютная ошибка, в долларах. **RMSE (Root Mean Squared Error)** — корень из средней квадратичной ошибки, тоже в долларах; большие ошибки влияют на неё сильнее.

**MAPE (Mean Absolute Percentage Error)** — средняя абсолютная процентная ошибка. Для каждого дома делим абсолютную ошибку на фактическую цену, затем усредняем:

$$\mathrm{MAPE}=\frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i-\widehat y_i}{y_i}\right|.$$

**Где:** $n$ — число проверяемых домов; $i$ — номер дома; $y_i$ — фактическая цена; $\widehat y_i$ — прогноз цены; $|\cdot|$ — модуль. Все $y_i$ должны быть ненулевыми. Множитель 100 переводит долю в проценты.

Например, ошибка 10 000 долларов для дома за 100 000 долларов — 10%, а для дома за 500 000 долларов — 2%. Поэтому одинаковые ошибки в долларах получают разный вес.

`mean_absolute_percentage_error` возвращает **долю**: 0,1 соответствует 10%. Для вывода в процентах умножаем результат на 100. При нулевом фактическом значении обычная формула не определена, а при очень маленьком — отдельная ошибка может сильно повлиять на результат. В нашей таблице цены положительны.

Сохраним три вычисления в небольшую функцию, чтобы одинаково оценивать все варианты. Она получает известные цены и прогнозы и возвращает метрики.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE, $": mean_absolute_error(y_true, y_pred),
        "RMSE, $": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE, %": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }

Повторим модель по трём площадям. Это будет точка отсчёта для сегодняшних изменений. Оцениваем её на прежней validation.

In [ ]:
base_model = LinearRegression()
base_model.fit(train[BASE_FEATURES], y_train)
base_pred = base_model.predict(validation[BASE_FEATURES])

base_metrics = regression_metrics(y_val, base_pred)
pd.Series(base_metrics).round(2)

### Как будем готовить признаки

Базовая модель готова. Дальше займёмся **предварительной обработкой данных (data preprocessing)**: заполнением пропусков, масштабированием числовых признаков и кодированием категорий. Это часть работы с признаками; создание и преобразование признаков часто называют **feature engineering**.

В scikit-learn для таких задач предусмотрены **классы-трансформеры** с общим интерфейсом: `fit`, `transform` и `fit_transform`. Разобравшись с этой логикой один раз, мы сможем одинаковым способом применять разные преобразования.

**`ColumnTransformer` (преобразователь столбцов)** объединяет обработку разных групп признаков в одном объекте. Например, к числовым столбцам можно применить инстументы заполнения пропусков и масштабирования, а к категориальным — инструменты векторизации. Он применяет заданные преобразования к выбранным столбцам и соединяет результаты в общую матрицу признаков. У него также есть `fit`, `transform` и `fit_transform`. Подробнее — в [официальном руководстве scikit-learn](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data) и [описании класса ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html).

Сначала разберём общую логику на схеме, затем выполним отдельные шаги и соберём их вместе.

In [ ]:
#@title Как работают fit, transform и fit_transform { display-mode: "form" }
from IPython.display import HTML, display
from uuid import uuid4

_transformer_html = r"""<div id="TM_OUTPUT_ROOT">
  <h2>fit: запомнить. transform: применить.</h2>
  <p><strong>Трансформер (transformer)</strong> — объект с методом <code>transform</code>, который преобразует данные. Трансформеры ниже сначала вычисляют параметры по train с помощью <code>fit</code>, затем применяют их к другим данным.</p>
  <div class="TM_OUTPUT_PREFIX-takeaway"><code>fit</code> — только на train. Для validation, test и новых объектов — <code>transform</code> с теми же параметрами.</div>
  <p class="text-small">Синтетический пример: четыре дома с площадями 50, 50, 150 и 150 м². Меняйте площадь нового дома и следите за результатом. Параметры обученного scaler остаются прежними.</p>
  <div class="viz-controls">
    <label class="form-label" for="TM_OUTPUT_PREFIX-area">Площадь нового дома: <output id="TM_OUTPUT_PREFIX-area-value" class="tabular-nums">175 м²</output>
      <input class="form-range" id="TM_OUTPUT_PREFIX-area" type="range" min="0" max="250" step="5" value="175">
    </label>
  </div>
  <svg id="TM_OUTPUT_PREFIX-flow" role="img" aria-label="fit запоминает среднее и стандартное отклонение train; transform использует их для нового дома"></svg>
  <div class="TM_OUTPUT_PREFIX-methods">
    <div><code>fit(train)</code><span>Вычислить и сохранить параметры по train → вернуть обученный объект</span></div>
    <div><code>transform(X)</code><span>Использовать сохранённые параметры → вернуть преобразованные данные</span></div>
    <div><code>fit_transform(train)</code><span>Для трансформеров ниже: fit(train), затем transform(train)</span></div>
  </div>
  <div id="TM_OUTPUT_PREFIX-live" class="sr-only" aria-live="polite"></div>

  <hr>
  <h3>Что сохраняем вместе с моделью</h3>
  <p><strong>Pipeline (конвейер обработки)</strong> объединяет шаги в цепочку. Здесь это подготовка признаков и модель. Сохраняем всю обученную цепочку: преобразователи вместе с весами модели.</p>
  <svg id="TM_OUTPUT_PREFIX-save" role="img" aria-label="Сохранённый Pipeline содержит обученные преобразователи, порядок признаков и веса модели. При pipeline.predict(X) новые данные проходят подготовку и поступают в модель."></svg>

  <hr>
  <h3>Какой трансформер — для каких признаков</h3>
  <div class="TM_OUTPUT_PREFIX-catalog">
    <section class="TM_OUTPUT_PREFIX-entry">
      <div><strong><a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html" target="_blank" rel="noopener">StandardScaler</a></strong><div class="text-small">Числа · стандартизация</div></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">fit → сохранённые параметры</span><span><code>mean_</code> = [μ<sub>train</sub>]<br><code>scale_</code> = [s<sub>train</sub>]</span></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">transform → результат</span><div class="TM_OUTPUT_PREFIX-formula">$$z=\frac{x-\mu_{\mathrm{train}}}{s_{\mathrm{train}}}$$</div></div>
      <div class="TM_OUTPUT_PREFIX-note text-small">μ<sub>train</sub> = mean(train); s<sub>train</sub> = std(train, ddof=0). Для преобразованного train: mean(z) ≈ 0, std(z, ddof=0) ≈ 1 при непостоянном столбце. Сохранённые параметры не меняются; нормальность распределения не гарантируется.</div>
    </section>
    <section class="TM_OUTPUT_PREFIX-entry">
      <div><strong><a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html" target="_blank" rel="noopener">MinMaxScaler</a></strong><div class="text-small">Числа · масштаб [0, 1]</div></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">fit → сохранённые параметры</span><span><code>data_min_</code> = [a<sub>train</sub>]<br><code>data_max_</code> = [b<sub>train</sub>]</span></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">transform → результат</span><div class="TM_OUTPUT_PREFIX-formula">$$z=\frac{x-a_{\mathrm{train}}}{b_{\mathrm{train}}-a_{\mathrm{train}}}$$</div></div>
      <div class="TM_OUTPUT_PREFIX-note text-small">a<sub>train</sub> = min(train), b<sub>train</sub> = max(train). [0, 1] — для train. Новые значения могут выйти за этот диапазон.</div>
    </section>
    <section class="TM_OUTPUT_PREFIX-entry">
      <div><strong><a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html" target="_blank" rel="noopener">RobustScaler</a></strong><div class="text-small">Числа · устойчивые центр и масштаб</div></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">fit → сохранённые параметры</span><span><code>center_</code> = [m<sub>train</sub>]<br><code>scale_</code> = [IQR<sub>train</sub>]</span></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">transform → результат</span><div class="TM_OUTPUT_PREFIX-formula">$$z=\frac{x-m_{\mathrm{train}}}{Q_3-Q_1}$$</div></div>
      <div class="TM_OUTPUT_PREFIX-note text-small">m<sub>train</sub> = median(train). Q₁ и Q₃ — 25-й и 75-й процентили train; IQR<sub>train</sub> = Q₃ − Q₁. Выбросы не удаляются.</div>
    </section>
    <section class="TM_OUTPUT_PREFIX-entry">
      <div><strong><a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html" target="_blank" rel="noopener">OneHotEncoder</a></strong><div class="text-small">Категории · индикаторы</div></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">fit → сохранённые параметры</span><span><code>categories_</code> = [[c₁, …, cₖ]]<br>Фиксированный порядок столбцов</span></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">transform → результат</span><div class="TM_OUTPUT_PREFIX-formula">$$z_j=[x=c_j]$$</div></div>
      <div class="TM_OUTPUT_PREFIX-note text-small">Скобка Айверсона [условие]: 1, если условие верно; иначе 0. Для одного признака k — число категорий в train, j = 1, …, k — номер категории, cⱼ — сама категория. Здесь сохраняем все k индикаторов; обработка новых категорий задаётся отдельно.</div>
    </section>
    <section class="TM_OUTPUT_PREFIX-entry">
      <div><strong><a href="https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html" target="_blank" rel="noopener">SimpleImputer</a></strong><div class="text-small">Пропуски · strategy="median"</div></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">fit → сохранённые параметры</span><span><code>statistics_</code> = [m<sub>train</sub>]<br>m<sub>train</sub> = median(train), без NaN</span></div>
      <div class="TM_OUTPUT_PREFIX-rule"><span class="TM_OUTPUT_PREFIX-phase">transform → результат</span><div class="TM_OUTPUT_PREFIX-formula">$$x^{\prime}=\begin{cases}x,&\text{нет пропуска},\\m_{\mathrm{train}},&\text{пропуск}.\end{cases}$$</div></div>
      <div class="TM_OUTPUT_PREFIX-note text-small">NaN — числовой пропуск. Числа: strategy="mean" / "median". Категории: "most_frequent" / "constant".</div>
    </section>
  </div>
  <div class="text-small TM_OUTPUT_PREFIX-glossary">Показаны атрибуты для одного признака; окончание <code>_</code> обозначает значения, полученные при <code>fit</code>. X — таблица признаков; x — одно исходное значение; z или x′ — преобразованное. Статистики считаются отдельно по столбцам train. Формулы скейлеров — для стандартных настроек и ненулевого масштаба; постоянные столбцы обрабатываются отдельно. StandardScaler использует ddof = 0.</div>
</div>
<style>
#TM_OUTPUT_ROOT {
  --foreground:#20252d; --muted-foreground:#626b78; --card:#f7f9fc;
  --border:#cdd5df; --background:#fff; --viz-series-1:#2874bc;
  box-sizing:border-box; width:100%; max-width:1000px; margin:16px auto;
  padding:20px; color:var(--foreground); background:var(--background);
  border:1px solid var(--border); border-radius:12px;
  font-family:system-ui,-apple-system,Segoe UI,sans-serif; font-size:16px; line-height:1.6;
}
#TM_OUTPUT_ROOT * { box-sizing:border-box; }
#TM_OUTPUT_ROOT h2 { font-size:25px; line-height:1.3; margin:0 0 16px; color:var(--foreground); }
#TM_OUTPUT_ROOT h3 { font-size:21px; line-height:1.35; margin:20px 0 12px; color:var(--foreground); }
#TM_OUTPUT_ROOT p { margin:12px 0; }
#TM_OUTPUT_ROOT hr { border:0; border-top:1px solid var(--border); margin:24px 0; }
#TM_OUTPUT_ROOT a { color:#1766aa; text-decoration:underline; }
#TM_OUTPUT_ROOT code { color:var(--foreground); background:var(--card); border-radius:4px; padding:2px 4px; font-size:.88em; }
#TM_OUTPUT_ROOT .text-small { font-size:14px; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-takeaway { padding:12px 16px; border-left:4px solid var(--viz-series-1); background:var(--card); margin:16px 0; }
#TM_OUTPUT_ROOT .viz-controls { margin:16px 0 8px; }
#TM_OUTPUT_ROOT .form-label { display:block; font-weight:500; }
#TM_OUTPUT_ROOT .form-range { display:block; width:100%; max-width:440px; margin:12px 0; accent-color:var(--viz-series-1); }
#TM_OUTPUT_ROOT input:focus-visible { outline:2px solid var(--viz-series-1); outline-offset:4px; }
#TM_OUTPUT_ROOT .tabular-nums { font-variant-numeric:tabular-nums; }
#TM_OUTPUT_ROOT .sr-only { position:absolute; width:1px; height:1px; padding:0; margin:-1px; overflow:hidden; clip:rect(0,0,0,0); white-space:nowrap; border:0; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-formula { font-size:17px; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-formula mjx-container[display="true"] { margin:8px 0!important; }
@media(prefers-color-scheme:dark) {
  #TM_OUTPUT_ROOT { --foreground:#e6e9ef; --muted-foreground:#b0bac8; --card:#1c2530; --border:#435164; --background:#121922; --viz-series-1:#8dc1ff; }
  #TM_OUTPUT_ROOT a { color:#8dc1ff; }
}
#TM_OUTPUT_ROOT svg[id$="-flow"], #TM_OUTPUT_ROOT svg[id$="-save"] { width:100%; display:block; overflow:visible; }
#TM_OUTPUT_ROOT svg[id$="-flow"] text, #TM_OUTPUT_ROOT svg[id$="-save"] text { fill:var(--foreground); font-family:inherit; font-size:14px; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-node { fill:var(--card); stroke:var(--border); stroke-width:1; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-memory { fill:color-mix(in srgb,var(--viz-series-1) 10%,var(--background)); stroke:var(--border); stroke-width:1; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-arrow { fill:none; stroke:var(--muted-foreground); stroke-width:1.5; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-methods { display:grid; gap:10px; margin:12px 0 20px; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-methods>div { display:grid; grid-template-columns:210px 1fr; gap:12px; align-items:start; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-catalog { display:grid; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-entry { display:grid; grid-template-columns:minmax(145px,0.8fr) minmax(185px,1fr) minmax(210px,1.1fr); gap:12px; padding:16px 0; border-bottom:1px solid var(--border); }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-rule { display:flex; flex-direction:column; gap:8px; justify-content:center; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-phase { color:var(--muted-foreground); }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-note { grid-column:2 / 4; }
#TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-glossary { margin-top:16px; }

@media(max-width:800px) {
  #TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-methods>div { grid-template-columns:1fr; gap:4px; }
  #TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-entry { grid-template-columns:1fr; gap:10px; }
  #TM_OUTPUT_ROOT .TM_OUTPUT_PREFIX-note { grid-column:auto; }
}
</style>
<script>
(() => {
  const root = document.getElementById('TM_OUTPUT_ROOT');
  const slider = root.querySelector('#TM_OUTPUT_PREFIX-area');
  const value = root.querySelector('#TM_OUTPUT_PREFIX-area-value');
  const svg = root.querySelector('#TM_OUTPUT_PREFIX-flow');
  const save = root.querySelector('#TM_OUTPUT_PREFIX-save');
  const ns = 'http://www.w3.org/2000/svg';
  const fmt = n => n.toLocaleString('ru-RU',{maximumFractionDigits:2});
  function el(parent,tag,attrs,text) {
    const node=document.createElementNS(ns,tag);
    Object.entries(attrs || {}).forEach(([k,v])=>node.setAttribute(k,v));
    if(text!==undefined) node.textContent=text;
    parent.appendChild(node); return node;
  }
  function setup(s,w,h,id) {
    s.replaceChildren(); s.setAttribute('viewBox','0 0 '+w+' '+h);s.style.height=h+'px';
    const defs=el(s,'defs');const marker=el(defs,'marker',{id,viewBox:'0 0 10 10',refX:9,refY:5,markerWidth:7,markerHeight:7,orient:'auto-start-reverse'});
    el(marker,'path',{d:'M 0 0 L 10 5 L 0 10 z',fill:'var(--muted-foreground)'});
  }
  function txt(s,x,y,lines,anchor='middle',weight='400') {
    lines.forEach((line,i)=>el(s,'text',{x,y:y+i*22,'text-anchor':anchor,'font-weight':weight},line));
  }
  function box(s,x,y,w,h,lines,memory=false) {
    el(s,'rect',{x,y,width:w,height:h,rx:6,'class':memory?'TM_OUTPUT_PREFIX-memory':'TM_OUTPUT_PREFIX-node'});
    const start=y+(h-(lines.length-1)*22)/2+4;
    txt(s,x+w/2,start,lines);
  }
  function arrow(s,d,id) {el(s,'path',{d,'class':'TM_OUTPUT_PREFIX-arrow','marker-end':'url(#'+id+')'});}
  function draw() {
    const w=Math.max(250,Math.floor(svg.getBoundingClientRect().width));
    const x=Number(slider.value);const z=(x-100)/50;
    value.textContent=fmt(x)+' м²';
    root.querySelector('#TM_OUTPUT_PREFIX-live').textContent='Площадь '+fmt(x)+' м²; результат '+fmt(z)+'. Сохранённые среднее 100 м² и стандартное отклонение 50 м² не изменились.';
    const wide=w>=720;
    setup(svg,w,wide?335:590,'TM_OUTPUT_PREFIX-arrow-main');
    if(wide) {
      const bw=(w-80)/3;
      const a=12,b=w/2-bw/2,c=w-bw-12;
      txt(svg,12,20,['1. Обучение на train'],'start','500');
      box(svg,a,42,bw,90,['Площадь, м²','50, 50, 150, 150']);
      box(svg,b,42,bw,90,['StandardScaler','mean_ = [100] м²','scale_ = [50] м²'],true);
      arrow(svg,'M '+(a+bw)+' 87 L '+(b-5)+' 87','TM_OUTPUT_PREFIX-arrow-main');
      txt(svg,(a+bw+b)/2,70,['fit']);
      txt(svg,c+bw/2,66,['Состояние сохранено','в объекте scaler']);
      txt(svg,12,190,['2. Применение к новым данным'],'start','500');
      box(svg,a,230,bw,76,['Новый дом',fmt(x)+' м²']);
      box(svg,b,230,bw,76,['scaler.transform','параметры из train']);
      box(svg,c,230,bw,76,['z = ('+fmt(x)+' − 100) / 50',fmt(z)+' · без единиц']);
      arrow(svg,'M '+(a+bw)+' 268 L '+(b-5)+' 268','TM_OUTPUT_PREFIX-arrow-main');
      arrow(svg,'M '+(b+bw)+' 268 L '+(c-5)+' 268','TM_OUTPUT_PREFIX-arrow-main');
      arrow(svg,'M '+(b+bw/2)+' 132 L '+(b+bw/2)+' 224','TM_OUTPUT_PREFIX-arrow-main');
      txt(svg,b+bw/2+12,165,['μ и s не меняются'],'start');
    } else {
      const bx=18,bw=w-36;
      txt(svg,18,20,['1. Обучение на train'],'start','500');
      box(svg,bx,38,bw,65,['Площадь, м²: 50, 50, 150, 150']);
      arrow(svg,'M '+w/2+' 103 L '+w/2+' 143','TM_OUTPUT_PREFIX-arrow-main');txt(svg,w/2+26,130,['fit'],'start');
      box(svg,bx,150,bw,92,['Обученный StandardScaler','mean_ = [100] м²','scale_ = [50] м²'],true);
      txt(svg,18,277,['2. Новые данные, прежние μ и s'],'start','500');
      box(svg,bx,298,bw,61,['Новый дом: '+fmt(x)+' м²']);
      arrow(svg,'M '+w/2+' 359 L '+w/2+' 393','TM_OUTPUT_PREFIX-arrow-main');
      box(svg,bx,400,bw,62,['scaler.transform','mean_ = [100], scale_ = [50]']);
      arrow(svg,'M 18 196 L 6 196 L 6 431 L 13 431','TM_OUTPUT_PREFIX-arrow-main');
      arrow(svg,'M '+w/2+' 462 L '+w/2+' 497','TM_OUTPUT_PREFIX-arrow-main');
      box(svg,bx,504,bw,64,['z = ('+fmt(x)+' − 100) / 50',fmt(z)+' · без единиц']);
    }
    setup(save,w,wide?255:410,'TM_OUTPUT_PREFIX-arrow-save');
    el(save,'rect',{x:8,y:8,width:w-16,height:wide?125:198,rx:6,'class':'TM_OUTPUT_PREFIX-memory'});
    txt(save,w/2,34,['Сохранённый обученный Pipeline'],'middle','500');
    const labels=[['Imputer','statistics_'],['Scaler','mean_, scale_'],['Encoder','categories_'],['Модель','coef_, intercept_']];
    if(wide){
      const step=(w-32)/4;
      labels.forEach((line,i)=>txt(save,16+step*(i+.5),70,line));
      txt(save,w/2,116,['+ список признаков, их порядок и настройки преобразований']);
      const bw=(w-100)/3,b=50+bw,c=88+2*bw;
      box(save,12,185,bw,56,['Новые данные']);
      box(save,b,185,bw,56,['pipeline.predict(X)','подготовка → прогноз']);
      box(save,c,185,bw,56,['Прогноз']);
      arrow(save,'M '+(12+bw)+' 213 L '+(b-5)+' 213','TM_OUTPUT_PREFIX-arrow-save');
      arrow(save,'M '+(b+bw)+' 213 L '+(c-5)+' 213','TM_OUTPUT_PREFIX-arrow-save');
      arrow(save,'M '+w/2+' 133 L '+w/2+' 179','TM_OUTPUT_PREFIX-arrow-save');
      txt(save,w/2+12,160,['загрузка'],'start');
    }else{
      labels.forEach((line,i)=>txt(save,23,62+i*29,[line[0]+': '+line[1]],'start'));
      txt(save,w/2,188,['+ порядок признаков и настройки']);
      arrow(save,'M '+w/2+' 206 L '+w/2+' 242','TM_OUTPUT_PREFIX-arrow-save');
      txt(save,w/2+12,232,['загрузка'],'start');
      box(save,18,249,w-36,55,['Новые данные: X']);
      arrow(save,'M '+w/2+' 304 L '+w/2+' 335','TM_OUTPUT_PREFIX-arrow-save');
      box(save,18,342,w-36,56,['pipeline.predict(X)','подготовка → прогноз']);
    }
  }
  slider.addEventListener('input',draw);
  let previousWidth = 0;
  new ResizeObserver(() => {
    const width = svg.getBoundingClientRect().width;
    if (Math.abs(width - previousWidth) > 0.5) { previousWidth = width; draw(); }
  }).observe(root);
  draw();

  function typeset() {
    if (window.MathJax?.typesetPromise) {
      window.MathJax.startup.promise.then(() => window.MathJax.typesetPromise([root]));
    } else if (window.MathJax?.Hub) {
      window.MathJax.Hub.Queue(['Typeset', window.MathJax.Hub, root]);
    }
  }
  const existingLoader = [...document.scripts].find(s => /mathjax/i.test(s.src));
  if (window.MathJax?.typesetPromise || window.MathJax?.Hub) {
    typeset();
  } else if (existingLoader) {
    existingLoader.addEventListener('load', typeset, {once:true});
    window.addEventListener('load', typeset, {once:true});
  } else {
    window.MathJax = window.MathJax || {tex:{inlineMath:[['$','$']],displayMath:[['$$','$$']]},svg:{fontCache:'local'}};
    const loader = document.createElement('script');
    loader.src = 'https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js';
    loader.onload = typeset;
    document.head.appendChild(loader);
  }
})();
</script>
"""
_transformer_id = "transformer-" + uuid4().hex
_transformer_html = _transformer_html.replace("TM_OUTPUT_ROOT", _transformer_id)
_transformer_html = _transformer_html.replace("TM_OUTPUT_PREFIX", _transformer_id + "-tm")
display(HTML(_transformer_html))

## 2. Что делать с пропусками?

**Пропущенное значение (missing value)** означает, что значение признака в записи отсутствует. В таблице числовой пропуск обычно представлен как `NaN`.

Добавим год постройки и `LotFrontage`. Сначала посчитаем пропуски на train.

In [ ]:
missing_counts = train[numeric_features].isna().sum()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=missing_counts.values, y=missing_counts.index, color="black", ax=ax)
ax.set_xlabel("Число пропущенных значений в train")
ax.set_ylabel("Признак")
plt.tight_layout()
plt.show()

print(missing_counts.to_string())

`LinearRegression` не принимает `NaN` в матрице признаков. Сегодня заполним числовые пропуски **медианой соответствующего столбца train**.

**Заполнение пропусков (imputation)** — замена отсутствующих значений по выбранному правилу. Медиана — простое начальное правило, а не восстановленная истинная длина участка. Это решение можно улучшать и проверять, как и саму модель.

> На train используем `fit_transform`. К validation и новым домам применяем только `transform`.

Если вычислить медиану по всей таблице, подготовка уже будет учитывать проверочные данные. Такое использование информации из проверки при обучении называют **утечкой данных (data leakage)**.

In [ ]:
imputer = SimpleImputer(strategy="median")

filled_train = pd.DataFrame(
    imputer.fit_transform(train[numeric_features]),
    columns=numeric_features,
    index=train.index,
)
filled_val = pd.DataFrame(
    imputer.transform(validation[numeric_features]),
    columns=numeric_features,
    index=validation.index,
)

pd.Series(imputer.statistics_, index=numeric_features, name="Медиана train")

In [ ]:
missing_ids = train.index[train["LotFrontage"].isna()][:5]

pd.DataFrame({
    "До заполнения, м": train.loc[missing_ids, "LotFrontage"],
    "После заполнения, м": filled_train.loc[missing_ids, "LotFrontage"],
})

Посмотрим, как изменился столбец. Слева — только известные длины, справа — весь train после заполнения. На месте медианы появится больше одинаковых значений: у нас появилась рабочая замена пропусков, но не новая измеренная информация.

In [ ]:
known_frontage = train["LotFrontage"].dropna()
frontage_median = imputer.statistics_[numeric_features.index("LotFrontage")]
bins = np.linspace(known_frontage.min(), known_frontage.max(), 25)

fig, axs = plt.subplots(figsize=(13, 4), ncols=2, sharey=True)
for ax, values, title in zip(
    axs,
    [known_frontage, filled_train["LotFrontage"]],
    ["Известные значения", "После заполнения пропусков"],
):
    ax.hist(values, bins=bins, color="black", alpha=0.65)
    ax.axvline(frontage_median, color="black", linestyle="--", label="Медиана train")
    ax.set_xlabel("Длина границы участка вдоль улицы LotFrontage, м")
    ax.set_ylabel("Количество домов")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

Теперь все пять столбцов числовые и заполнены. Обучим регрессию привычным способом.

In [ ]:
numeric_model = LinearRegression()
numeric_model.fit(filled_train, y_train)
numeric_pred = numeric_model.predict(filled_val)
numeric_metrics = regression_metrics(y_val, numeric_pred)

pd.DataFrame([base_metrics, numeric_metrics],
             index=["Три площади", "Пять числовых признаков"]).round(2)

## 3. Приводим признаки к сопоставимому масштабу

В одной таблице соседствуют площадь, год постройки и длина участка. Их числовые значения имеют разный масштаб.

**Стандартизация (standardization)** — преобразование, при котором из значения признака вычитают среднее по train и делят результат на стандартное отклонение по train:

$$z=\frac{x-\mu_{\mathrm{train}}}{s_{\mathrm{train}}}.$$

**Где:** $x$ — значение одного признака; $\mu_{\mathrm{train}}$ — его среднее на train; $s_{\mathrm{train}}$ — его стандартное отклонение на train; $z$ — преобразованное безразмерное значение. Статистики считаются отдельно для каждого столбца, здесь — после заполнения пропусков.

В `sklearn` это делает **`StandardScaler`**. На train среднее станет примерно 0, а стандартное отклонение — 1. `StandardScaler` использует стандартное отклонение с делением на число объектов, то есть `ddof=0`. Если столбец постоянный, библиотека не делит на ноль и использует масштаб 1. На validation среднее не обязано быть равно 0: там используем статистики, вычисленные на train.

Стандартизация помогает работе градиентных методов и важна для регуляризации, которую разберём дальше. **Она не превращает распределение в нормальное и не делает признаки независимыми.**

In [ ]:
scaler = StandardScaler()
scaled_train = pd.DataFrame(
    scaler.fit_transform(filled_train), columns=numeric_features, index=train.index
)
scaled_val = pd.DataFrame(
    scaler.transform(filled_val), columns=numeric_features, index=validation.index
)

pd.DataFrame({
    "Среднее до": filled_train.mean(),
    "Стд. отклонение до": filled_train.std(ddof=0),
    "Среднее после": scaled_train.mean(),
    "Стд. отклонение после": scaled_train.std(ddof=0),
}).round(3)

Посмотрим на связь жилой площади с ценой до и после стандартизации. Дома и цены остались теми же. Изменились начало отсчёта и масштаб горизонтальной оси.

In [ ]:
fig, axs = plt.subplots(figsize=(13, 5), ncols=2)
axs[0].scatter(filled_train["GrLivArea"], y_train, alpha=0.2, color="black")
axs[0].set_xlabel("Надземная жилая площадь GrLivArea, м²")
axs[0].set_title("До стандартизации")
axs[1].scatter(scaled_train["GrLivArea"], y_train, alpha=0.2, color="black")
axs[1].set_xlabel("Стандартизованная жилая площадь, безразмерное z")
axs[1].set_title("После стандартизации")
for ax in axs:
    ax.set_ylabel("Стоимость продажи SalePrice, доллары США")
plt.tight_layout()
plt.show()

### Улучшатся ли прогнозы обычного МНК?

У линейной модели со свободным членом изменение масштаба можно компенсировать изменением коэффициентов. Для обычного МНК при полном столбцовом ранге расширенной матрицы признаков со столбцом единиц стандартизация сама по себе не должна менять прогнозы, кроме численной погрешности. Проверим это на наших данных.

In [ ]:
scaled_model = LinearRegression()
scaled_model.fit(scaled_train, y_train)
scaled_pred = scaled_model.predict(scaled_val)

print("Максимальная разница прогнозов, доллары США:",
      np.max(np.abs(numeric_pred - scaled_pred)))
pd.DataFrame([
    numeric_metrics,
    regression_metrics(y_val, scaled_pred),
], index=["Без стандартизации", "Со стандартизацией"]).round(2)

## 4. Собираем шаги в Pipeline

Сейчас подготовка выполняется несколькими отдельными объектами. Для новых домов нужно повторить те же преобразования и в том же порядке.

**Pipeline (конвейер обработки)** — объект, который объединяет последовательные шаги. Он может состоять только из преобразователей. Если последним шагом стоит модель, цепочка также делает прогноз. Сейчас соберём именно такой вариант.

```text
Числовые признаки → заполнение пропусков → стандартизация → регрессия
```

В списке шагов каждой операции даём имя. Имена `imputer`, `scaler`, `regression` выбираем сами; после имени записываем объект, выполняющий действие.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regression", LinearRegression()),
])

numeric_pipeline.fit(train[numeric_features], y_train)
pipeline_pred = numeric_pipeline.predict(validation[numeric_features])

print("Совпадают ли прогнозы с пошаговым расчётом:",
      np.allclose(pipeline_pred, scaled_pred))

Когда вызываем `numeric_pipeline.fit`, она:

1. запоминает медианы train и заполняет пропуски;
2. запоминает средние и стандартные отклонения полученной таблицы и масштабирует её;
3. обучает коэффициенты регрессии.

Когда вызываем `predict`, цепочка использует **уже запомненные** медианы, средние, масштабы и коэффициенты. Обучения заново не происходит.

> В `fit` передаём только train. Сам по себе Pipeline не исправит ошибку, если обучить его сразу на всех данных.

## 5. Как передать модели категориальный признак?

**Категориальный признак (categorical feature)** — признак, принимающий значения из набора категорий. Например, название района. Также категориальный признак часто представлен в формате `string` что не поддерживается большинством моделей `scikit learn`. Также, если мы видим признак, который представлен в виде `{1, 2, 3, 4}` и в контексте данных мы понимаем что это не вещественная величина а категории (номера групп) - то использовать этот признак в таком виде для линейной модели - очень плохая идея. 

Если просто заменить районы числами 1, 2, 3, линейная модель начнёт трактовать эти коды как величины с порядком и одинаковыми шагами. Названия районов такой мы не можем наделять количественным смыслоы - это заставит модель выучивать потенциально ложные закономерности. 

Процедупа ревращения категориального признака в вид, приемлемый для моделирования "непрерывными параметрическими моделями" (к которым относятся и линейные) называется `encoding` или `vectorizing` (векторизация). Существует большое количество разных подходов векторизации, но сегодня мы посмотрим на самый простой - `One-Hot Encoding` - он не лишен недостатков, но наиболее удобен для первого знакомства.

**One-Hot Encoding (кодирование фиктивной (бинарной) матрицей)**: для каждой категории создадим столбец, в котором 1 означает принадлежность к категории, а 0 — отсутствие принадлежности. Сначала посмотрим на маленький пример.

In [ ]:
toy_districts = pd.DataFrame({"Район": ["Центр", "Север", "Юг", "Центр"]},
                             index=["Дом 1", "Дом 2", "Дом 3", "Дом 4"])
toy_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
toy_encoded = pd.DataFrame(
    toy_encoder.fit_transform(toy_districts),
    columns=toy_encoder.get_feature_names_out(), index=toy_districts.index,
)
toy_encoded

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(toy_encoded, annot=True, fmt=".0f", cmap="Greys", vmin=0, vmax=1,
            linewidths=1, cbar=False, ax=ax)
ax.set_xlabel("Индикаторы районов: 1 — принадлежит, 0 — не принадлежит")
ax.set_ylabel("Объект")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

У кодировщика тот же интерфейс: `fit` запоминает категории train, `transform` создаёт столбцы по запомненному списку. `sparse_output=False` просит вернуть обычный числовой массив, удобный для просмотра в небольшой учебной задаче.

Что произойдёт, если придёт новый район? При **`handle_unknown="ignore"`** его индикаторы станут нулевыми, и преобразование сможет продолжиться. Это технический способ обработать новую категорию; данных о её влиянии на цену у модели не появится.

In [ ]:
toy_new = pd.DataFrame({"Район": ["Центр", "Запад"]})
pd.DataFrame(toy_encoder.transform(toy_new),
             columns=toy_encoder.get_feature_names_out(), index=toy_new["Район"])

### Связь с линейной зависимостью

В примере у каждого дома ровно один район, поэтому сумма всех трёх индикаторов равна 1:

$$I_{\text{Север}}+I_{\text{Центр}}+I_{\text{Юг}}=1.$$

**Где:** $I_{\text{Север}}, I_{\text{Центр}}, I_{\text{Юг}}$ — индикаторы соответствующих районов для одного дома; каждый равен 0 или 1.

Сумма столбцов индикаторов совпадает со столбцом единиц для свободного члена. Получается точная линейная зависимость, которую мы обсуждали в выводе формулы МНК.

В модели с `LinearRegression` уберём один индикатор параметром **`drop="first"`**. Оставшиеся нули будут означать опорную категорию — первую в списке кодировщика. При неизменных числовых признаках коэффициент другого района будет задавать отличие прогноза от опорного района.

`LinearRegression` умеет подбирать прогнозы и при зависимых столбцах, но отдельные коэффициенты тогда не определяются единственным образом. Удаление одного индикатора устраняет именно эту зависимость; другие связи между признаками могут сохраняться.

При сочетании `drop="first"` и `handle_unknown="ignore"` новый район тоже получит нулевой код, как опорный. Это ограничение выбранной схемы, а не доказательство одинаковых цен в этих районах.

## 6. Разным столбцам — разная подготовка

Числовые признаки нужно заполнить и масштабировать, а район — закодировать. **ColumnTransformer (преобразователь столбцов)** применяет разные преобразования к указанным группам столбцов, затем объединяет результаты в одну матрицу.

```text
Числовые признаки → заполнение → стандартизация ┐
                                              ├→ матрица признаков → регрессия
Район            → One-Hot Encoding            ┘
```

`ColumnTransformer` отвечает за разные ветви подготовки. Внешний `Pipeline` соединит подготовку с моделью. Соберём эти два объекта по очереди.

В выбранном `Neighborhood` на train нет пропусков. Поэтому для него сегодня достаточно кодировщика. Пропуски в числовых столбцах обрабатывает `SimpleImputer`.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
     categorical_features),
])

Каждая строка внутри `ColumnTransformer` задаёт три вещи: **имя ветви, преобразователь, список столбцов**. Он соберёт сначала результат ветви `num`, затем `cat`; остальные столбцы по умолчанию отбросит.

Добавим регрессию и обучим всю цепочку на train.

In [ ]:
model = Pipeline([
    ("preprocess", preprocessor),
    ("regression", LinearRegression()),
])

model.fit(train[features], y_train)
print("Подготовка и модель обучены на train.")

Посмотрим, какую матрицу получает регрессия. `named_steps` позволяет обратиться к шагу по имени, а `get_feature_names_out` возвращает названия получившихся столбцов. Префиксы `num__` и `cat__` указывают ветвь подготовки.

In [ ]:
fitted_preprocessor = model.named_steps["preprocess"]
prepared_train = fitted_preprocessor.transform(train[features])
prepared_columns = fitted_preprocessor.get_feature_names_out()

print("Исходная таблица признаков:", train[features].shape)
print("Подготовленная матрица:", prepared_train.shape)
pd.DataFrame(prepared_train[:5], columns=prepared_columns, index=train.index[:5])

### Новые категории встречаются и в реальных данных

Проверим, есть ли в validation районы, отсутствующие в train. Категории кодировщика уже зафиксированы; дообучать его на validation не будем.

In [ ]:
known_districts = set(train["Neighborhood"])
new_districts = sorted(set(validation["Neighborhood"]) - known_districts)
print("Новые районы в validation:", new_districts)

При следующем вызове библиотека может вывести предупреждение о неизвестной категории и её нулевом коде. Это описанное выше поведение `handle_unknown="ignore"`: прогноз вычисляется, но новый район не имеет собственного обученного коэффициента.

In [ ]:
full_pred = model.predict(validation[features])
full_metrics = regression_metrics(y_val, full_pred)

results = pd.DataFrame([
    {"Модель": "Три площади", **base_metrics},
    {"Модель": "Пять числовых признаков", **numeric_metrics},
    {"Модель": "Числовые признаки и район", **full_metrics},
])
results.round(2)

In [ ]:
fig, axs = plt.subplots(figsize=(16, 4), ncols=3)
for ax, metric, label in zip(
    axs,
    ["MAE, $", "RMSE, $", "MAPE, %"],
    ["MAE на validation, доллары США", "RMSE на validation, доллары США", "MAPE на validation, %"],
):
    sns.barplot(data=results, x=metric, y="Модель", color="black", ax=ax)
    ax.set_xlabel(label)
    ax.set_ylabel("Модель")
plt.tight_layout()
plt.show()

**Обсудим:** помог ли район по выбранной основной метрике MAE? Согласуются ли с этим RMSE и MAPE?

Разные метрики могут по-разному ранжировать модели: у них разные способы учитывать ошибки. Поэтому критерий выбора мы определили до сравнения. Универсального значения MAPE, которое делает любую модель «хорошей», нет.

Это проверка на одном фиксированном разбиении. В следующих занятиях разберём, как оценивать устойчивость результата.

## 7. Прогноз для нового дома

Передадим исходные признаки: район всё ещё строка, а длина границы участка неизвестна. Внутри `predict` пройдут те же подготовленные шаги.

Как и при обучении, площади передаём в м², а `LotFrontage` — в метрах. Перевод из футов мы выполняли при загрузке таблицы; он не входит в этот Pipeline.

In [ ]:
new_house = pd.DataFrame({
    "GrLivArea": [120.0],
    "GarageArea": [25.0],
    "TotalBsmtSF": [80.0],
    "YearBuilt": [1998],
    "LotFrontage": [np.nan],
    "Neighborhood": ["NAmes"],
})
new_price = model.predict(new_house)[0]
print(f"Прогноз стоимости: {new_price:,.0f} долларов США")

## 8. Самостоятельная работа

### Задача 1. Какая медиана попадёт в новый объект?

В маленьком train известны длины 10, 20 и 30 м, а одно значение пропущено. Пришли два новых объекта: у одного значение пропущено, у другого длина 100 м.

1. Запишите, чем должны заполниться пропуски.
2. Проверьте ответ с `SimpleImputer(strategy="median")`: обучите его только на маленьком train.
3. Объясните, почему нельзя заново вызвать `fit_transform` на двух новых объектах.

In [ ]:
small_train = pd.DataFrame({"LotFrontage": [10.0, 20.0, np.nan, 30.0]})
small_new = pd.DataFrame({"LotFrontage": [np.nan, 100.0]})

# TODO: fit an imputer on small_train and transform both tables.

**Ваш ответ:** …

### Задача 2. Добавьте площадь участка

Добавьте `LotArea` к числовым признакам полной модели. Площадь уже переведена в м² в начале ноутбука.

1. Создайте новую цепочку: заполнение и стандартизация чисел, кодирование района, регрессия.
2. Обучите её на прежнем train и получите прогнозы для прежней validation.
3. Посчитайте MAE, RMSE и MAPE. Для MAPE используйте проценты.
4. Добавьте результат в копию таблицы `results` и сравните с моделью без `LotArea`.

Можно использовать предыдущий код как образец. Основную метрику и разбиение не меняем; test остаётся отложенным.

In [ ]:
my_numeric_features = numeric_features + ["LotArea"]
my_features = my_numeric_features + categorical_features

# TODO: build and fit a new preprocessing/model pipeline.
# TODO: predict validation prices, calculate three metrics, and compare results.

**Ваш вывод:** …

### Задача 3. Объясните получившееся решение

* Какой вариант вы выбираете по MAE на validation? Укажите все три метрики выбранной модели.
* Что запомнили `SimpleImputer`, `StandardScaler`, `OneHotEncoder` и сама регрессия во время `fit`?
* Почему возможность вычислить прогноз для нового района ещё не означает, что модель знает особенности этого района?

**Ваш ответ:** …

## Что мы теперь умеем

У нас есть модель, которой можно передать исходную строку с числовыми признаками, пропуском и названием района. Подготовка выполняется в нужном порядке и использует статистики только из train.

**Числа → заполнение и стандартизация. Категории → кодирование. Затем общая матрица → линейная регрессия.**

На следующем занятии вернёмся к коэффициентам: посмотрим, почему они могут быть нестабильными и как помогает регуляризация.

## Источники

* [House Prices: данные и описание признаков](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).
* [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) — заполнение пропусков.
* [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) — стандартизация и её параметры.
* [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) и [RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html) — альтернативные масштабы признаков.
* [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) — категории, `drop` и `handle_unknown`.
* [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) и [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html).
* [MAPE в scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html) — доли, проценты и поведение при нулях.
* [Типичные ошибки и утечка данных](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).
* [Сохранение обученных объектов](https://scikit-learn.org/stable/model_persistence.html) — подготовка данных и модель.